# GPU Hardware: Memory, Compute, and Interconnects

> Kernel layouts, matrix instructions, and thread roles depend on GPU storage and compute structure. Weights move from HBM into on-chip SRAM before Tensor Cores perform multiply-adds; cross-GPU results travel through NVLink, PCIe, or a network. The slowest segment often determines end-to-end speed.
>
> **Inside one GPU**, HBM stores models and caches, while small high-bandwidth SRAM holds the current tile; SMs and Tensor Cores execute parallel work. **Between GPUs**, NVLink/NVSwitch connect devices within a node, PCIe connects host and devices, and InfiniBand or RoCE connects nodes.
>
> Hardware feeds back into **parallel strategy**. Communication-heavy Tensor Parallelism usually stays inside fast node interconnects, while Pipeline or Data Parallelism can cross nodes more readily.

We begin with HBM and SRAM, then expand to SMs, Tensor Cores, NVLink, PCIe, and network fabrics to see how hardware constraints shape kernels and distributed training.


## 0. GPU Memory Hierarchy

A GPU has storage at very different scales. **HBM** (High Bandwidth Memory) is large enough for the model but relatively distant and slower. **SRAM** is a small, extremely fast on-chip store for working intermediates.

Capacity and speed trade off physically, so hardware places size in HBM and speed in SRAM. Performance often depends less on arithmetic rate than on how often data moves between them.


### HBM and SRAM

- **HBM** is what we commonly call GPU memory: tens of gigabytes, large but comparatively slower.
- **SRAM** is on-chip storage: only tens of megabytes across an H100, but much higher bandwidth.

Remember: **HBM makes data fit; SRAM makes computation fast.**


## 1. HBM and SRAM Capacity

The following official specifications show recent data-center GPUs. Do not memorize exact values; observe that both capacity and speed increase while the division of labor remains one large/slower store and one small/faster store.


In [ ]:
# === Memory configurations of mainstream data-center GPUs ===
# Source: official NVIDIA specification tables
# Each row: model, HBM capacity GB, HBM bandwidth TB/s, SRAM capacity MB, release year

gpus = [
    ("A100 40GB",  40, 1.55,  20, 2020),
    ("A100 80GB",  80, 2.00,  20, 2021),
    ("H100 SXM",   80, 3.35,  20, 2022),
    ("H200 SXM",  141, 4.80,  20, 2024),
    ("B200",      192, 8.00,  40, 2024),
]

print(f"{'Model':<14} {'HBM(GB)':>9} {'HBM BW(TB/s)':>15} {'SRAM(MB)':>10} {'Year':>6}")
print("-" * 60)
for name, hbm_gb, hbm_bw, sram_mb, year in gpus:
    print(f"{name:<14} {hbm_gb:>9} {hbm_bw:>15.2f} {sram_mb:>10} {year:>6}")

print()
print("Key observation 1: HBM capacity grows from 40 GB to 192 GB, nearly fivefold.")
print("Key observation 2: SRAM remains only 20-40 MB; on-chip cache is always scarce.")
print("HBM determines whether the model fits; effective SRAM use determines how fast it runs.")


### 1.1 HBM and SRAM Bandwidth

Bandwidth measures bytes moved per second. H100 HBM reaches about 3.35 TB/s, while aggregate on-chip SRAM bandwidth is often estimated near 30 TB/s—roughly an order of magnitude higher.

Repeated HBM traffic therefore bottlenecks an operation. Keeping reused data in SRAM is a core idea behind optimizations such as FlashAttention.


In [ ]:
# === HBM versus SRAM bandwidth using H100 ===
hbm_bw_tb = 3.35    # H100 HBM bandwidth in TB/s
sram_bw_tb = 30.0   # estimated H100 SRAM bandwidth in TB/s

ratio = sram_bw_tb / hbm_bw_tb
print(f"H100 HBM bandwidth:  {hbm_bw_tb:.2f} TB/s")
print(f"H100 SRAM bandwidth: {sram_bw_tb:.2f} TB/s, estimated")
print(f"Gap: SRAM is {ratio:.1f}x HBM")
print()
print("Key observation: repeatedly moving data from HBM makes a computation limited by HBM bandwidth;")
print("Keeping data in SRAM can expose roughly ten times the bandwidth.")


### 1.2 SRAM Reuse in Attention

Attention forms a score matrix of sequence length $N$ by $N$. For $N=8192$, 32 heads, and BF16 values, calculate the storage and movement required by this intermediate.


In [ ]:
# === Hand-calculate the size of the N×N Attention score matrix ===
N = 8192        # sequence length
num_heads = 32  # number of heads
bf16 = 2        # bytes per BF16 value

# Score shape is [heads,N,N], containing num_heads*N*N values
num_elements = num_heads * N * N
total_bytes = num_elements * bf16

print(f"Score-matrix shape: [{num_heads}, {N}, {N}]")
print(f"Element count:       {num_elements:,}")
print(f"Bytes:               {total_bytes / 1e9:.2f} GB")
print()
# Naive implementation writes this matrix to HBM and reads it back for softmax
# One write plus one read means two transfers
round_trip_bytes = 2 * total_bytes
hbm_bw = 3.35e12  # H100 HBM bandwidth: 3.35 TB/s

t = round_trip_bytes / hbm_bw
print(f"Naive implementation writes once and reads once, moving {round_trip_bytes / 1e9:.2f} GB")
print(f"HBM takes {t * 1000:.2f} ms just for this matrix and this work.")
print()
print("Key observation: a modest increase in N makes this matrix grow quadratically.")
print("FlashAttention tiles the matrix and completes each tile in SRAM,")
print("avoiding a full N×N write to HBM and making use of SRAM bandwidth.")


### 1.3 HBM vs. SRAM Summary

- HBM is large and comparatively slow; SRAM is small and fast.
- SRAM bandwidth is roughly an order of magnitude higher.
- High-performance kernels keep intermediates in SRAM whenever possible; FlashAttention is a representative example.

Next we examine the compute side.


## 2. GPU Compute Units

- An **SM** (Streaming Multiprocessor) is a major GPU execution unit with local on-chip storage. An H100 has many SMs.
- A **warp** is the scheduling group of 32 threads that execute together.

You do not need to write CUDA to use these concepts when reasoning about parallelism and occupancy.


In [ ]:
# === H100 compute resources: SM count and size ===
sm_count = 132                  # number of streaming multiprocessors
shared_mem_per_sm_kb = 256      # shared-memory size in each SM
register_per_sm_kb = 256        # register capacity in each SM

print(f"H100 SM count:                {sm_count}")
print(f"Shared memory per SM:         {shared_mem_per_sm_kb} KB")
print(f"Registers per SM:             {register_per_sm_kb} KB")
print(f"Total shared memory:          {sm_count * shared_mem_per_sm_kb / 1024:.1f} MB")
print()
print(f"Warp size is fixed at 32 threads across NVIDIA architectures.")
print(f"Key observation: total SRAM is about 33 MB, matching the 20-30 MB scale introduced earlier.")


Neural networks rely heavily on matrix multiplication. **Tensor Cores** are specialized units for small matrix multiply-accumulate operations and provide far higher throughput than general-purpose cores.

Numerical format affects both movement and throughput. FP16/BF16 use 2 bytes; FP8 uses 1 byte, halving storage and traffic while compatible Tensor Cores offer higher arithmetic throughput. The trade-off is reduced range and precision, so FP8 training uses scaling to keep values representable.


FP8 has two complementary formats:

| Format | Sign | Exponent | Mantissa | Typical role |
|:---|---:|---:|---:|:---|
| E4M3 | 1 | 4 | 3 | Forward weights/activations: more precision, less range |
| E5M2 | 1 | 5 | 2 | Backward gradients: more range, less precision |

Compared with BF16, FP8 halves bytes and bandwidth demand, but requires scale management because its numerical range is smaller.


## 3. GPU Interconnects

One GPU cannot hold or efficiently train every model. Multi-GPU speed depends on the paths between devices:

| Path | Purpose | Scope |
|:---|:---|:---|
| PCIe | General host/device bus | Within a node |
| NVLink | High-bandwidth direct GPU interconnect | Within a node |
| NVSwitch | Switch fabric connecting many GPUs | Within a node |
| InfiniBand / RoCE | High-performance network | Across nodes |

We compare their bandwidth before calculating transfer times.


In [ ]:
# === Multi-GPU interconnect bandwidth scales ===
# Values from public NVIDIA specifications
# GB/s means billions of bytes per second; larger is faster

interconnects = [
    # name, bandwidth GB/s, scope, common use
    ("PCIe 4.0 x16",      32,  "intra-node",  "GPU-CPU, storage, low-end GPU-GPU"),
    ("PCIe 5.0 x16",      64,  "intra-node",  "H100 to CPU"),
    ("NVLink 3.0",       300,  "intra-node",  "direct A100 links"),
    ("NVLink 4.0",       450,  "intra-node",  "direct H100 links"),
    ("NVLink 5.0",       900,  "intra-node",  "direct B200 links"),
    ("InfiniBand NDR",    50,  "inter-node",  "400 Gbps, one link"),
    ("InfiniBand XDR",   100,  "inter-node",  "800 Gbps, one link"),
]

print(f"{'Name':<20} {'Bandwidth(GB/s)':>12} {'Scope':>10}  Typical use")
print("-" * 80)
for name, bw, scope, use in interconnects:
    print(f"{name:<20} {bw:>12} {scope:>8}  {use}")

print()
print("Key observation 1: within a node, NVLink is 6-10 times faster than PCIe.")
print("Key observation 2: cross-node InfiniBand is 4-9 times slower than intra-node NVLink.")
print("Two GPUs in one machine communicate much faster than two GPUs in different machines.")


### 3.1 Transfer Time for a 70B Model

A 70B BF16 model occupies about 140 GB. Calculate how long moving that volume would take over NVLink and PCIe. Similar-scale transfers arise during parameter, gradient, or model-state movement.


In [ ]:
# === Hand-calculate moving a 70B model over NVLink versus PCIe ===
P = 70e9          # 70 billion model parameters
bf16_bytes = 2    # two bytes per BF16 parameter
param_bytes = P * bf16_bytes   # about 140 GB

nvlink_bw = 450    # bidirectional NVLink 4.0 GB/s
pcie_bw = 64       # PCIe 5.0 x16 GB/s

t_nvlink = param_bytes / (nvlink_bw * 1e9)
t_pcie = param_bytes / (pcie_bw * 1e9)

print(f"70B BF16 parameters total {param_bytes / 1e9:.0f} GB")
print(f"Over NVLink 4.0: {t_nvlink:.2f} seconds")
print(f"Over PCIe 5.0:  {t_pcie:.2f} seconds")
print(f"Speed gap: {t_pcie / t_nvlink:.1f}x")
print()
print("Key observation: identical data takes seven times longer when it uses a slower interconnect.")
print("Every gradient synchronization moves data at this scale, so the wrong path makes every step about seven times slower.")


### 3.2 NVLink and PCIe

PCIe is a general bus shared by many device types and carries protocol and controller overhead. NVLink is designed specifically for GPU-to-GPU communication and bypasses much of that path, offering much higher within-node bandwidth.

Exact bandwidth depends on generation and topology. The durable rule is: **prefer NVLink for frequent within-node GPU communication.**


Connecting every pair of eight GPUs directly would require 28 point-to-point links. **NVSwitch** provides a switching fabric: each GPU connects to switches, and any GPU pair communicates through the fabric. Multi-switch H100 systems provide high-bandwidth all-to-all connectivity without a dedicated physical cable for every pair.


Large clusters also need cross-node communication. **InfiniBand** and RoCE provide high bandwidth and low latency compared with ordinary networking. A node may expose one network path per GPU, and GPU Direct RDMA can move data directly between GPU memory across nodes without staging through CPU memory.

The next calculation estimates a 70B transfer over a 400 Gbps (50 GB/s) link.


In [ ]:
# === Hand-calculate moving a 70B model across nodes ===
ib_ndr_gbps = 400          # one IB NIC in Gbps
ib_ndr_gbs = ib_ndr_gbps / 8   # convert to GB/s; eight bits per byte
P = 70e9
param_bytes = P * 2        # BF16

n_hca = 8                  # commonly eight IB NICs per machine, one per GPU
agg_bw = n_hca * ib_ndr_gbs   # aggregate bandwidth across eight NICs

t_single = param_bytes / (ib_ndr_gbs * 1e9)
t_agg = param_bytes / (agg_bw * 1e9)

print(f"One IB link: {ib_ndr_gbs:.0f} GB/s; eight-link aggregate: {agg_bw:.0f} GB/s")
print(f"Move a 70B model across nodes ({param_bytes / 1e9:.0f} GB):")
print(f"  one link:  {t_single:.2f} seconds")
print(f"  eight links: {t_agg:.2f} seconds")
print()
print("For comparison, intra-node NVLink moves the same data in only 0.31 seconds.")
print("Key observation: even with all eight IB links, cross-node transfer remains more than twice as slow as intra-node transfer.")


## 4. DGX H100 Hardware Structure

A DGX H100 combines eight H100 SXM GPUs, an NVSwitch fabric for within-node GPU connectivity, high-speed network adapters for cross-node traffic, and CPUs for control and data loading.

The essential pattern is: **NVSwitch inside the node, InfiniBand or RoCE between nodes.**


In [ ]:
# === Intra-node versus inter-node aggregate bandwidth ===
# Intra-node: NVLink 450 GB/s between every pair, C(8,2)=28 pairs
intra_link = 450
n_pairs = 8 * 7 // 2        # 28 pairs
intra_agg = intra_link * n_pairs

# Inter-node: eight 50 GB/s IB NICs
ib_link = 50
n_hca = 8
inter_agg = ib_link * n_hca

print(f"Intra-node full-mesh bandwidth: {intra_agg:,} GB/s, 28 pairs × 450")
print(f"Inter-node aggregate bandwidth: {inter_agg} GB/s, eight links × 50")
print()
print(f"A {intra_agg / inter_agg:.1f}x gap.")
print()
print("Key observation: this 30-fold gap underlies every parallel-strategy choice.")
print("Keep communication-heavy operations within a node and minimize cross-node traffic.")


## 5. Hardware Constraints and Training Strategy

Within-node NVLink bandwidth can be many times higher than one cross-node network link. Parallel methods communicate differently:

- **Tensor Parallelism** synchronizes within every layer and should usually remain inside an NVLink domain.
- **Pipeline Parallelism** mainly sends activations between neighboring stages and can cross nodes more readily.
- **Data Parallelism / FSDP** synchronizes gradients or shards per step and can use aggregated network bandwidth.
- **Expert Parallelism** routes tokens to expert-owning GPUs and is especially topology-sensitive.

The durable rule is: **place the largest and most frequent communication on the highest-bandwidth links.**


In [ ]:
# === Bandwidth sensitivity of parallel strategies ===
strategies = [
# strategy, communication pattern, frequency, topology requirement
    ("Tensor Parallelism",   "synchronize every layer", "very frequent", "must be intra-node NVLink"),
    ("Pipeline Parallelism", "pass adjacent activations", "rare",      "can cross nodes"),
    ("DDP / FSDP",           "sync gradients each step", "infrequent", "intra- or inter-node"),
    ("MoE Expert Parallelism", "all-to-all each step", "frequent",     "prefer intra-node"),
]

print(f"{'Strategy':<24}{'Communication':<24}{'Frequency':<14}Topology")
print("-" * 78)
for s, mode, freq, req in strategies:
    print(f"{s:<24}{mode:<20}{freq:<12}{req}")
print()
print("Key observation: more frequent communication creates stronger dependence on high-speed links such as NVLink.")


## 6. Inspecting GPU Topology

When multi-GPU training is unexpectedly slow, inspect topology first with `nvidia-smi topo -m`. Its matrix describes the path between every GPU pair. The notebook uses a representative eight-GPU output; run the command directly on real hardware.


In [ ]:
# === Example nvidia-smi topo -m output for an eight-GPU H100 node ===
# On a real machine, run: nvidia-smi topo -m
# Here a string simulates the typical output

topo_output = """
        GPU0    GPU1    GPU2    GPU3    GPU4    GPU5    GPU6    GPU7
GPU0     X      NV12    NV12    NV12    NV12    NV12    NV12    NV12
GPU1    NV12     X      NV12    NV12    NV12    NV12    NV12    NV12
GPU2    NV12    NV12     X      NV12    NV12    NV12    NV12    NV12
GPU3    NV12    NV12    NV12     X      NV12    NV12    NV12    NV12
GPU4    NV12    NV12    NV12    NV12     X      NV12    NV12    NV12
GPU5    NV12    NV12    NV12    NV12    NV12     X      NV12    NV12
GPU6    NV12    NV12    NV12    NV12    NV12    NV12     X      NV12
GPU7    NV12    NV12    NV12    NV12    NV12    NV12    NV12     X

Legend:
  X    = Self
  SYS  = Across NUMA nodes (slowest)
  PIX  = Same PCIe switch (uses PCIe, slower)
  PHB  = Across PCIe Host Bridges (slower)
  NV#  = Number of NVLink lanes (fast)
"""

print(topo_output)
print("How to read it:")
print("  NV12 everywhere means all eight GPUs have 12 NVLink lanes between them: healthy and fast.")
print("  Large PIX or SYS regions mean NVLink is unavailable, so communication falls back to slower PCIe.")
print()
print("Key observation: this matrix is the basis of topology-aware scheduling:")
print("the scheduler places communication-heavy work on GPUs connected by NV12.")


### 6.1 Common Hardware Inspection Commands

- `nvidia-smi`: memory, temperature, and utilization per GPU.
- `watch -n 1 nvidia-smi`: refresh once per second.
- `nvtop`: interactive GPU monitoring.
- NCCL tests such as `all_reduce_perf`: measure actual inter-GPU bandwidth; values far below specification suggest topology, driver, or firmware problems.


## Summary

| Concept | Main point |
|:---|:---|
| HBM | Large capacity, lower bandwidth than SRAM |
| SRAM | Small on-chip working storage with high bandwidth |
| SM / Warp | GPU execution organization |
| Tensor Core | Lower-precision matrix acceleration |
| PCIe | General but slower device bus |
| NVLink | Fast within-node GPU path |
| NVSwitch | High-bandwidth all-to-all within a node |
| InfiniBand / RoCE | Cross-node networking |

- [ ] I can explain why keeping intermediates in SRAM matters.
- [ ] I understand how lower precision affects Tensor Core throughput.
- [ ] I can distinguish PCIe, NVLink, NVSwitch, and cross-node networks.
- [ ] I know why communication-heavy parallelism should remain within fast topology domains.
- [ ] I can use `nvidia-smi topo -m` as a first topology diagnostic.


**Exercise 1: Calculate an Attention Score Matrix**

For $N=8192$, 32 heads, and BF16, calculate the `[h,N,N]` matrix size and two HBM transfers (write then read).

Hint: elements $=hN^2$; multiply by 2 bytes and again by 2 transfers.


In [ ]:
# Exercise 1: Attention score-matrix size and transfer volume
N = 8192
h = 32
bf16_bytes = 2

# TODO: total memory (GB)
mat_gb = None

# TODO: total two-way traffic in GB, one write plus one read
round_trip_gb = None

assert mat_gb is not None, 'Please replace the placeholder before running the assertion.'
assert round_trip_gb is not None, 'Please replace the placeholder before running the assertion.'

expected_mat = h * N * N * bf16_bytes / 1e9
assert abs(mat_gb - expected_mat) < 0.1, f"Matrix should be {expected_mat:.2f} GB"
assert abs(round_trip_gb - 2 * expected_mat) < 0.2, f"Traffic should be {2 * expected_mat:.2f} GB"

print(f"Exercise 1 passed:")
print(f"  score matrix: {mat_gb:.2f} GB")
print(f"  write plus read: {round_trip_gb:.2f} GB")
print(f"  This is one layer and one forward pass; longer sequences make SRAM residency increasingly important.")


**Exercise 2: Within-Node vs. Cross-Node Transfer**

Transfer 140 GB of BF16 parameters over NVLink at 450 GB/s and over a 50 GB/s network link. Calculate both times and their ratio.

Hint: time equals bytes divided by bandwidth.


In [ ]:
# Exercise 2: same-node versus cross-node transfer time
P = 70e9
param_bytes = P * 2       # BF16

nvlink_bw = 450           # bidirectional NVLink 4.0 GB/s
ib_bw = 50                # one IB NDR link in GB/s

# TODO: two transfer times in seconds
t_intra = None
t_inter = None

assert t_intra is not None and t_inter is not None, 'Please replace the placeholder before running the assertion.'
expected_intra = param_bytes / (nvlink_bw * 1e9)
expected_inter = param_bytes / (ib_bw * 1e9)
assert abs(t_intra - expected_intra) < 0.01, f"Intra-node should be {expected_intra:.2f} seconds"
assert abs(t_inter - expected_inter) < 0.01, f"Inter-node should be {expected_inter:.2f} seconds"

print(f"Exercise 2 passed:")
print(f"  intra-node NVLink: {t_intra:.2f} seconds")
print(f"  inter-node IB:     {t_inter:.2f} seconds")
print(f"  A {t_inter / t_intra:.1f}x gap, which is why communication-heavy work must remain within a node.")


**Exercise 3: Read a Topology Matrix**

An eight-GPU `nvidia-smi topo -m` output shows `PIX` rather than NVLink paths. What happens to DDP performance, and what should be checked first?

Hint: gradient synchronization now travels over PCIe; inspect NVLink enablement, drivers, firmware, CUDA compatibility, and platform configuration.


In [ ]:
# Exercise 3: identify a topology issue

# Symptom: select one of three
#   "training speed is normal and matches NVLink"
#   "training slows sharply because all-reduce synchronization becomes the bottleneck"
#   "training fails to start"
phenomenon = None

# Diagnostic direction: select one of three
#   "whether the driver and CUDA support NVLink and the motherboard enables it"
#   "increase batch size"
#   "replace one GPU"
check_action = None

assert phenomenon == "training slows sharply because all-reduce synchronization becomes the bottleneck", \
    "PIX means PCIe, which is 6-10 times slower than NVLink, so gradient synchronization becomes the bottleneck"
assert check_action == "whether the driver and CUDA support NVLink and the motherboard enables it", \
    "If NVLink is not working, inspect the driver, CUDA, and motherboard configuration first"

print(f"Exercise 3 passed:")
print(f"  symptom: {phenomenon}")
print(f"  inspect: {check_action}")


## References

- NVIDIA, [H100 Tensor Core GPU Architecture Whitepaper](https://resources.nvidia.com/en-us/tensor-core), 2022
- NVIDIA, [Hopper FP8 Tensor Cores](https://www.nvidia.com/en-us/data-center/hopper-architecture/), 2022
- NVIDIA, [NVLink and NVSwitch](https://www.nvidia.com/en-us/data-center/nvlink/), 2023
- NVIDIA, [DGX H100 System Architecture](https://www.nvidia.com/en-us/data-center/dgx-h100/), 2023
- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), 2022
- NVIDIA, [NCCL Tests Documentation](https://github.com/NVIDIA/nccl-tests), 2023